# 01 · Lakehouse data profiling

Profiles every Delta table in the attached Lakehouse (or the one named in `LAKEHOUSE_ROOT`) and persists
the results as Delta tables so the downstream notebooks can reason about the data without re-scanning it.

**What it produces** (in the `OUTPUT_SCHEMA` schema, or with an `OUTPUT_SCHEMA_` prefix on non-schema lakehouses):

| Table | Grain | Key content |
|---|---|---|
| `table_profile` | one row per source table | row count, size, column count, best candidate primary key, date / measure column counts |
| `column_profile` | one row per source column | type, null %, distinct count, uniqueness, min / max / avg, string lengths, sample values, inferred semantic role, candidate-key flag |
| `relationship_candidates` | one row per (child column → parent key) pair tested | name-match reason, value containment %, orphan count, inferred cardinality |
| `profiling_runs` | one row per run | parameters and timing, so later runs can be compared |

**How to use it**
1. Attach the Lakehouse you want to understand as the default Lakehouse (or set `LAKEHOUSE_ROOT`).
2. Adjust the parameters cell (schemas / table patterns to include, exactness, relationship scan mode).
3. Run all. Review the summary sections at the bottom, then move on to `02_logical_model_design`.

The profile is computed with a single aggregation pass per table plus one small sample, so it scales to
large tables. Relationship discovery is bounded to name-matched column pairs unless `RELATIONSHIP_SCAN_MODE = "all"`.

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
LAKEHOUSE_ROOT = ""            # abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id> ; "" = default lakehouse
SOURCE_SCHEMAS = ""            # comma-separated schemas to profile ("" = every schema; ignored on non-schema lakehouses)
INCLUDE_TABLES = ""            # comma-separated glob patterns, e.g. "sales_*,customer*" ("" = all tables)
EXCLUDE_TABLES = "profiling*,dim_*,fact_*,gold*"   # glob patterns to skip (pipeline outputs are skipped by default)
OUTPUT_SCHEMA = "profiling"    # where results land; becomes a table-name prefix on non-schema lakehouses
SAMPLE_ROWS = 5                # example values captured per column
EXACT_DISTINCT = False         # True = exact distinct counts on every table (slower on very large tables)
MAX_ROWS_FOR_EXACT = 5000000   # tables at or below this size always get exact distinct counts
RELATIONSHIP_SCAN_MODE = "name"  # "name" = value-check only name-matched pairs; "all" = value-check every type-compatible pair (expensive)
MIN_CONTAINMENT = 0.95         # share of child values that must exist in the parent key to flag a relationship candidate
MAX_KEY_DISTINCT_RATIO = 1.0   # candidate key must have distinct == non-null rows (leave at 1.0 for strict)
RUN_ID = ""                    # "" = auto-generated timestamp id

In [ ]:
import re, json, fnmatch, datetime as dt, time
from pyspark.sql import functions as F, types as T


def _parse_list(s):
    return [x.strip() for x in (s or "").split(",") if x.strip()]


def _lakehouse_root():
    if LAKEHOUSE_ROOT:
        return LAKEHOUSE_ROOT.rstrip("/")
    ctx = notebookutils.runtime.context
    ws, lh = ctx.get("defaultLakehouseWorkspaceId"), ctx.get("defaultLakehouseId")
    if not lh:
        raise RuntimeError("Attach a default Lakehouse to this notebook or set LAKEHOUSE_ROOT.")
    return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"


def _is_delta_dir(path):
    try:
        return any(f.name.rstrip("/") == "_delta_log" for f in notebookutils.fs.ls(path))
    except Exception:
        return False


def discover_tables(root):
    """Return (schema_enabled, [(schema, table, path)]) by walking <root>/Tables.
    Schema-enabled lakehouses hold Tables/<schema>/<table>/_delta_log; classic ones hold Tables/<table>/_delta_log."""
    tables_dir = f"{root}/Tables"
    entries = [e for e in notebookutils.fs.ls(tables_dir) if e.isDir]
    found, schema_enabled = [], False
    for e in entries:
        name = e.name.rstrip("/")
        if _is_delta_dir(e.path):
            found.append(("", name, e.path.rstrip("/")))
        else:
            for t in notebookutils.fs.ls(e.path):
                if t.isDir and _is_delta_dir(t.path):
                    schema_enabled = True
                    found.append((name, t.name.rstrip("/"), t.path.rstrip("/")))
    return schema_enabled, found


ROOT = _lakehouse_root()
RUN_ID = RUN_ID or dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
SCHEMA_ENABLED, ALL_TABLES = discover_tables(ROOT)


def output_path(name):
    """Delta folder for a profiling output table. Fabric auto-registers Delta folders under Tables/."""
    if SCHEMA_ENABLED and OUTPUT_SCHEMA:
        return f"{ROOT}/Tables/{OUTPUT_SCHEMA}/{name}"
    prefix = f"{OUTPUT_SCHEMA}_" if OUTPUT_SCHEMA else ""
    return f"{ROOT}/Tables/{prefix}{name}"


def output_ref(name):
    """Catalog name of a profiling output table, for Spark SQL / SQL endpoint use."""
    if SCHEMA_ENABLED and OUTPUT_SCHEMA:
        return f"{OUTPUT_SCHEMA}.{name}"
    return f"{OUTPUT_SCHEMA}_{name}" if OUTPUT_SCHEMA else name


inc, exc, schemas = _parse_list(INCLUDE_TABLES), _parse_list(EXCLUDE_TABLES), _parse_list(SOURCE_SCHEMAS)
TABLES = []
for schema, table, path in ALL_TABLES:
    if schemas and schema not in schemas:
        continue
    full = f"{schema}.{table}" if schema else table
    if inc and not any(fnmatch.fnmatch(table, p) or fnmatch.fnmatch(full, p) for p in inc):
        continue
    if exc and any(fnmatch.fnmatch(table, p) or fnmatch.fnmatch(full, p) for p in exc):
        continue
    TABLES.append((schema, table, path))

print(f"Run id: {RUN_ID}")
print(f"Lakehouse root: {ROOT}")
print(f"Schema-enabled lakehouse: {SCHEMA_ENABLED}")
print(f"Tables discovered: {len(ALL_TABLES)}, selected for profiling: {len(TABLES)}")
for s, t, _ in TABLES:
    print("  -", f"{s}.{t}" if s else t)

## Column-level profiling

One aggregation pass per table computes, for every column: non-null count, distinct count (exact or
approximate depending on size), min / max, mean and standard deviation for numerics, and string length
statistics. A small `limit()` sample supplies example values. Each column then receives an inferred
**semantic role** used by the modelling notebook:

| Role | Rule of thumb |
|---|---|
| `key` | unique, non-null, and named like an identifier (`*_id`, `*_key`, `*_code`, `id`) |
| `identifier` | named like an identifier but not unique (typically a foreign key) |
| `date` | date / timestamp type, or an integer named like `*_date` / `*_dt` |
| `flag` | boolean, or at most two distinct values |
| `measure` | numeric, not an identifier, and varied enough to be additive |
| `category` | low-cardinality string (≤ 50 distinct, or ≤ 1 % of rows) |
| `text` | high-cardinality string |
| `attribute` | anything else (numeric codes with low cardinality, etc.) |

In [ ]:
NUMERIC = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
TEMPORAL = (T.DateType, T.TimestampType)
ID_PATTERN = re.compile(r"(^id$|_id$|id$|_key$|key$|_code$|_no$|_num$|_number$|_sk$|_pk$|_fk$)", re.I)
DATE_NAME = re.compile(r"(date|_dt$|^dt_|timestamp|_ts$)", re.I)
MEASURE_NAME = re.compile(r"(amount|amt|price|cost|revenue|sales|total|qty|quantity|count|value|balance|fee|tax|discount|weight|hours|units|score|rate|pct|percent)", re.I)


def _q(c):
    return F.col(f"`{c}`")


def _fmt(v):
    if v is None:
        return None
    if isinstance(v, float):
        return f"{v:.6g}"
    return str(v)[:200]


def classify_column(name, dtype, row_count, non_null, distinct, is_candidate_key):
    n = name.lower()
    if isinstance(dtype, TEMPORAL) or (DATE_NAME.search(n) and isinstance(dtype, (T.IntegerType, T.LongType, T.StringType))):
        return "date"
    if isinstance(dtype, T.BooleanType) or (non_null > 0 and distinct <= 2 and row_count > 10):
        return "flag"
    if is_candidate_key and (ID_PATTERN.search(n) or isinstance(dtype, (T.IntegerType, T.LongType))):
        return "key"
    if ID_PATTERN.search(n):
        return "identifier"
    if isinstance(dtype, NUMERIC):
        if MEASURE_NAME.search(n):
            return "measure"
        # numeric with many distinct values and not an id → measure; low cardinality numeric → attribute
        return "measure" if (non_null and distinct / max(non_null, 1) > 0.01 and distinct > 20) else "attribute"
    if isinstance(dtype, T.StringType):
        if is_candidate_key:
            return "key"
        return "category" if (distinct <= 50 or distinct <= 0.01 * max(row_count, 1)) else "text"
    return "attribute"


def profile_table(schema, table, path):
    t0 = time.time()
    df = spark.read.format("delta").load(path)
    row_count = df.count()
    exact = EXACT_DISTINCT or row_count <= MAX_ROWS_FOR_EXACT
    fields = df.schema.fields
    aggs = []
    for f in fields:
        c, col = f.name, _q(f.name)
        aggs.append(F.count(col).alias(f"{c}__nn"))
        aggs.append((F.countDistinct(col) if exact else F.approx_count_distinct(col)).alias(f"{c}__nd"))
        if isinstance(f.dataType, NUMERIC + TEMPORAL + (T.StringType,)):
            aggs += [F.min(col).alias(f"{c}__min"), F.max(col).alias(f"{c}__max")]
        if isinstance(f.dataType, NUMERIC):
            aggs += [F.avg(col).alias(f"{c}__avg"), F.stddev(col).alias(f"{c}__std"),
                     F.sum(F.when(col < 0, 1).otherwise(0)).alias(f"{c}__neg"),
                     F.sum(F.when(col == 0, 1).otherwise(0)).alias(f"{c}__zero")]
        if isinstance(f.dataType, T.StringType):
            aggs += [F.avg(F.length(col)).alias(f"{c}__avglen"), F.max(F.length(col)).alias(f"{c}__maxlen"),
                     F.sum(F.when(F.trim(col) == "", 1).otherwise(0)).alias(f"{c}__empty")]
    stats = df.agg(*aggs).first().asDict() if fields else {}
    sample = [r.asDict() for r in df.limit(max(SAMPLE_ROWS * 4, 20)).collect()]

    try:
        detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").first()
        size_bytes, num_files = int(detail["sizeInBytes"]), int(detail["numFiles"])
    except Exception:
        size_bytes, num_files = None, None

    cols = []
    for i, f in enumerate(fields):
        c = f.name
        nn = int(stats.get(f"{c}__nn") or 0)
        nd = int(stats.get(f"{c}__nd") or 0)
        null_pct = round(100.0 * (row_count - nn) / row_count, 2) if row_count else 0.0
        uniq = round(nd / nn, 4) if nn else 0.0
        is_key = bool(row_count > 0 and nn == row_count and nd >= nn * MAX_KEY_DISTINCT_RATIO and exact)
        if not exact and row_count > 0 and nn == row_count and nd >= nn * 0.98:
            is_key = True  # approx distinct: within HLL error of unique
        values = []
        for r in sample:
            v = r.get(c)
            if v is not None and _fmt(v) not in values:
                values.append(_fmt(v))
            if len(values) >= SAMPLE_ROWS:
                break
        cols.append({
            "run_id": RUN_ID, "schema_name": schema, "table_name": table, "column_name": c, "ordinal": i,
            "data_type": f.dataType.simpleString(), "nullable": f.nullable, "row_count": row_count,
            "non_null_count": nn, "null_count": row_count - nn, "null_pct": null_pct,
            "distinct_count": nd, "distinct_is_exact": exact, "uniqueness_ratio": uniq,
            "empty_string_count": int(stats.get(f"{c}__empty") or 0) if isinstance(f.dataType, T.StringType) else None,
            "min_value": _fmt(stats.get(f"{c}__min")), "max_value": _fmt(stats.get(f"{c}__max")),
            "avg_value": float(stats[f"{c}__avg"]) if stats.get(f"{c}__avg") is not None else None,
            "stddev_value": float(stats[f"{c}__std"]) if stats.get(f"{c}__std") is not None else None,
            "negative_count": int(stats.get(f"{c}__neg") or 0) if isinstance(f.dataType, NUMERIC) else None,
            "zero_count": int(stats.get(f"{c}__zero") or 0) if isinstance(f.dataType, NUMERIC) else None,
            "avg_length": float(stats[f"{c}__avglen"]) if stats.get(f"{c}__avglen") is not None else None,
            "max_length": int(stats[f"{c}__maxlen"]) if stats.get(f"{c}__maxlen") is not None else None,
            "sample_values": " | ".join(values),
            "is_candidate_key": is_key,
            "is_constant": bool(nd <= 1 and row_count > 1),
            "semantic_role": classify_column(c, f.dataType, row_count, nn, nd, is_key),
        })

    keys = [x for x in cols if x["is_candidate_key"]]
    # best PK: prefer id-like names, then integer types, then lowest ordinal
    keys.sort(key=lambda x: (0 if ID_PATTERN.search(x["column_name"]) else 1, 0 if x["data_type"] in ("int", "bigint") else 1, x["ordinal"]))
    tbl = {
        "run_id": RUN_ID, "schema_name": schema, "table_name": table, "table_path": path,
        "row_count": row_count, "column_count": len(fields), "size_bytes": size_bytes, "num_files": num_files,
        "candidate_pk": keys[0]["column_name"] if keys else None,
        "candidate_key_columns": ",".join(k["column_name"] for k in keys),
        "date_column_count": sum(1 for x in cols if x["semantic_role"] == "date"),
        "measure_column_count": sum(1 for x in cols if x["semantic_role"] == "measure"),
        "identifier_column_count": sum(1 for x in cols if x["semantic_role"] in ("identifier", "key")),
        "text_column_count": sum(1 for x in cols if x["semantic_role"] in ("category", "text")),
        "high_null_column_count": sum(1 for x in cols if x["null_pct"] >= 50),
        "constant_column_count": sum(1 for x in cols if x["is_constant"]),
        "profile_seconds": round(time.time() - t0, 1), "profiled_at": dt.datetime.utcnow(),
    }
    return tbl, cols


table_rows, column_rows = [], []
for schema, table, path in TABLES:
    label = f"{schema}.{table}" if schema else table
    try:
        t, c = profile_table(schema, table, path)
        table_rows.append(t); column_rows.extend(c)
        print(f"profiled {label:40s} rows={t['row_count']:>12,} cols={t['column_count']:>4} pk={t['candidate_pk']} ({t['profile_seconds']}s)")
    except Exception as ex:  # keep going — one bad table must not abort the run
        print(f"FAILED  {label}: {ex}")

## Relationship discovery

Candidate parent keys are the columns flagged `is_candidate_key`. For every other table, each column is
paired with those keys when the **names** suggest a link (same name, `<parent_singular>_id`, a suffix match
such as `ship_customer_id` → `customer_id`, or `customers.id` ← `orders.customer_id`). Each pair is then
**value-checked** with one semi-join: the share of child *rows* whose value exists in the parent key
(`containment_rows`), the share of distinct child *values* that do (`containment`), and the orphan counts.
Pairs whose row containment is at or above `MIN_CONTAINMENT` become relationship candidates, so a handful of
orphan rows (typical late-arriving or deleted parents) does not hide a real relationship.

Set `RELATIONSHIP_SCAN_MODE = "all"` to value-check every type-compatible pair instead of only name matches.
This finds relationships with unhelpful column names but costs one join per pair.

In [ ]:
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())


def _singular(word):
    w = word.lower()
    for pre in ("dim_", "fact_", "tbl_", "stg_", "raw_", "src_", "bronze_", "silver_", "t_", "v_"):
        if w.startswith(pre):
            w = w[len(pre):]
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith(("ses", "xes", "zes", "ches", "shes")):
        return w[:-2]
    if w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def _type_family(dtype):
    if dtype in ("int", "bigint", "smallint", "tinyint") or dtype.startswith("decimal(") and dtype.endswith(",0)"):
        return "int"
    if dtype in ("string", "varchar", "char") or dtype.startswith("varchar") or dtype.startswith("char"):
        return "str"
    if dtype in ("date", "timestamp"):
        return "temporal"
    return dtype


def name_match_reason(child_table, child_col, parent_table, parent_col):
    cc, pc = _norm(child_col), _norm(parent_col)
    ps = _norm(_singular(parent_table))
    if cc == pc and cc not in ("id", "key", "code"):
        return "same_name"
    if pc in ("id", "key", "code") and cc in (ps + pc, ps + "id", ps + "key"):
        return "parent_singular_plus_" + pc
    if cc in (ps + "id", ps + "key", ps + "code", ps + "sk", ps + "fk"):
        return "parent_singular_id"
    if len(pc) >= 5 and cc.endswith(pc) and cc != pc:
        return "suffix_match"
    return None


col_by_table = {}
for c in column_rows:
    col_by_table.setdefault((c["schema_name"], c["table_name"]), []).append(c)
# Parents: strict candidate keys, plus "near keys" (id-like names, ≥ NEAR_KEY_UNIQUENESS unique) so a handful of
# duplicate rows in a dimension source does not hide the relationship. Near keys are flagged parent_key_is_unique = False.
NEAR_KEY_UNIQUENESS = 0.95
parent_keys = [c for c in column_rows if c["row_count"] > 0 and
               (c["is_candidate_key"] or (c["semantic_role"] == "identifier" and c["uniqueness_ratio"] >= NEAR_KEY_UNIQUENESS))]

pairs = []
for (cs, ct), cols in col_by_table.items():
    for cc in cols:
        if cc["non_null_count"] == 0:
            continue
        for pk in parent_keys:
            if (pk["schema_name"], pk["table_name"]) == (cs, ct):
                continue
            if _type_family(cc["data_type"]) != _type_family(pk["data_type"]):
                continue
            reason = name_match_reason(ct, cc["column_name"], pk["table_name"], pk["column_name"])
            if reason or RELATIONSHIP_SCAN_MODE == "all":
                pairs.append((cc, pk, reason or "value_scan"))

print(f"Candidate parent keys: {len(parent_keys)}  |  column pairs to value-check: {len(pairs)}")
if RELATIONSHIP_SCAN_MODE == "all" and len(pairs) > 500:
    print("WARNING: value-scanning >500 pairs; consider RELATIONSHIP_SCAN_MODE='name' or narrowing INCLUDE_TABLES.")

path_of = {(t["schema_name"], t["table_name"]): t["table_path"] for t in table_rows}
_df_cache = {}


def _table_df(schema, table):
    key = (schema, table)
    if key not in _df_cache:
        _df_cache[key] = spark.read.format("delta").load(path_of[key])
    return _df_cache[key]


rel_rows = []
for cc, pk, reason in pairs:
    child_all = _table_df(cc["schema_name"], cc["table_name"]).select(_q(cc["column_name"]).alias("v")).where(F.col("v").isNotNull())
    parent = _table_df(pk["schema_name"], pk["table_name"]).select(_q(pk["column_name"]).alias("v")).where(F.col("v").isNotNull()).distinct()
    # one join, then count matched rows and matched distinct values from the same result
    matched = child_all.join(parent, "v", "left_semi").agg(F.count("v").alias("rows"), F.countDistinct("v").alias("distinct")).first()
    child_rows, child_distinct = cc["non_null_count"], (cc["distinct_count"] if cc["distinct_is_exact"] else child_all.distinct().count())
    matched_rows, matched_distinct = int(matched["rows"]), int(matched["distinct"])
    containment_rows = round(matched_rows / child_rows, 4) if child_rows else 0.0        # share of child ROWS that find a parent
    containment = round(matched_distinct / child_distinct, 4) if child_distinct else 0.0  # share of child VALUES that find a parent
    child_is_unique = cc["is_candidate_key"]
    rel_rows.append({
        "run_id": RUN_ID,
        "child_schema": cc["schema_name"], "child_table": cc["table_name"], "child_column": cc["column_name"],
        "parent_schema": pk["schema_name"], "parent_table": pk["table_name"], "parent_column": pk["column_name"],
        "match_reason": reason, "child_distinct": child_distinct, "matched_distinct": matched_distinct,
        "orphan_distinct": child_distinct - matched_distinct, "containment": containment,
        "child_rows": child_rows, "orphan_rows": child_rows - matched_rows, "containment_rows": containment_rows,
        "child_null_pct": cc["null_pct"],
        "parent_key_is_unique": bool(pk["is_candidate_key"]),
        "cardinality": ("one_to_one" if child_is_unique else "many_to_one"),
        "is_candidate": bool(containment_rows >= MIN_CONTAINMENT and child_rows > 0),
    })

rel_rows.sort(key=lambda r: (-r["is_candidate"], -r["containment_rows"], r["child_table"]))
print(f"Relationship candidates found: {sum(r['is_candidate'] for r in rel_rows)} of {len(rel_rows)} pairs tested")

## Persist results
Results are appended with `run_id` so successive profiling runs can be compared. Downstream notebooks read the latest run by default.

In [ ]:
TABLE_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()), T.StructField("schema_name", T.StringType()), T.StructField("table_name", T.StringType()),
    T.StructField("table_path", T.StringType()), T.StructField("row_count", T.LongType()), T.StructField("column_count", T.IntegerType()),
    T.StructField("size_bytes", T.LongType()), T.StructField("num_files", T.IntegerType()), T.StructField("candidate_pk", T.StringType()),
    T.StructField("candidate_key_columns", T.StringType()), T.StructField("date_column_count", T.IntegerType()),
    T.StructField("measure_column_count", T.IntegerType()), T.StructField("identifier_column_count", T.IntegerType()),
    T.StructField("text_column_count", T.IntegerType()), T.StructField("high_null_column_count", T.IntegerType()),
    T.StructField("constant_column_count", T.IntegerType()), T.StructField("profile_seconds", T.DoubleType()),
    T.StructField("profiled_at", T.TimestampType()),
])
COLUMN_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()), T.StructField("schema_name", T.StringType()), T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()), T.StructField("ordinal", T.IntegerType()), T.StructField("data_type", T.StringType()),
    T.StructField("nullable", T.BooleanType()), T.StructField("row_count", T.LongType()), T.StructField("non_null_count", T.LongType()),
    T.StructField("null_count", T.LongType()), T.StructField("null_pct", T.DoubleType()), T.StructField("distinct_count", T.LongType()),
    T.StructField("distinct_is_exact", T.BooleanType()), T.StructField("uniqueness_ratio", T.DoubleType()),
    T.StructField("empty_string_count", T.LongType()), T.StructField("min_value", T.StringType()), T.StructField("max_value", T.StringType()),
    T.StructField("avg_value", T.DoubleType()), T.StructField("stddev_value", T.DoubleType()), T.StructField("negative_count", T.LongType()),
    T.StructField("zero_count", T.LongType()), T.StructField("avg_length", T.DoubleType()), T.StructField("max_length", T.IntegerType()),
    T.StructField("sample_values", T.StringType()), T.StructField("is_candidate_key", T.BooleanType()),
    T.StructField("is_constant", T.BooleanType()), T.StructField("semantic_role", T.StringType()),
])
REL_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()), T.StructField("child_schema", T.StringType()), T.StructField("child_table", T.StringType()),
    T.StructField("child_column", T.StringType()), T.StructField("parent_schema", T.StringType()), T.StructField("parent_table", T.StringType()),
    T.StructField("parent_column", T.StringType()), T.StructField("match_reason", T.StringType()), T.StructField("child_distinct", T.LongType()),
    T.StructField("matched_distinct", T.LongType()), T.StructField("orphan_distinct", T.LongType()), T.StructField("containment", T.DoubleType()),
    T.StructField("child_rows", T.LongType()), T.StructField("orphan_rows", T.LongType()), T.StructField("containment_rows", T.DoubleType()),
    T.StructField("child_null_pct", T.DoubleType()), T.StructField("parent_key_is_unique", T.BooleanType()),
    T.StructField("cardinality", T.StringType()), T.StructField("is_candidate", T.BooleanType()),
])
RUN_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()), T.StructField("lakehouse_root", T.StringType()), T.StructField("schema_enabled", T.BooleanType()),
    T.StructField("tables_profiled", T.IntegerType()), T.StructField("columns_profiled", T.IntegerType()),
    T.StructField("pairs_tested", T.IntegerType()), T.StructField("relationship_candidates", T.IntegerType()),
    T.StructField("parameters", T.StringType()), T.StructField("started_at", T.TimestampType()), T.StructField("finished_at", T.TimestampType()),
])

params = {k: globals()[k] for k in ["LAKEHOUSE_ROOT", "SOURCE_SCHEMAS", "INCLUDE_TABLES", "EXCLUDE_TABLES", "OUTPUT_SCHEMA", "SAMPLE_ROWS",
                                    "EXACT_DISTINCT", "MAX_ROWS_FOR_EXACT", "RELATIONSHIP_SCAN_MODE", "MIN_CONTAINMENT"]}
run_row = [{"run_id": RUN_ID, "lakehouse_root": ROOT, "schema_enabled": SCHEMA_ENABLED, "tables_profiled": len(table_rows),
            "columns_profiled": len(column_rows), "pairs_tested": len(rel_rows), "relationship_candidates": sum(r["is_candidate"] for r in rel_rows),
            "parameters": json.dumps(params), "started_at": min([t["profiled_at"] for t in table_rows] or [dt.datetime.utcnow()]),
            "finished_at": dt.datetime.utcnow()}]

outputs = {"table_profile": (table_rows, TABLE_SCHEMA), "column_profile": (column_rows, COLUMN_SCHEMA),
           "relationship_candidates": (rel_rows, REL_SCHEMA), "profiling_runs": (run_row, RUN_SCHEMA)}
for name, (rows, schema) in outputs.items():
    df = spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").save(output_path(name))
    print(f"wrote {len(rows):>6,} rows → {output_ref(name)}   ({output_path(name)})")

## Summary — what the profile says about this Lakehouse
Review these before running `02_logical_model_design`. Each section is a plain DataFrame, so it can be filtered or exported.

In [ ]:
tp = spark.createDataFrame([tuple(r.get(f.name) for f in TABLE_SCHEMA.fields) for r in table_rows], TABLE_SCHEMA)
cp = spark.createDataFrame([tuple(r.get(f.name) for f in COLUMN_SCHEMA.fields) for r in column_rows], COLUMN_SCHEMA)
rp = spark.createDataFrame([tuple(r.get(f.name) for f in REL_SCHEMA.fields) for r in rel_rows], REL_SCHEMA)

print("=== Tables (largest first) ===")
display(tp.select("schema_name", "table_name", "row_count", "column_count", "size_bytes", "candidate_pk",
                  "date_column_count", "measure_column_count", "identifier_column_count", "high_null_column_count", "constant_column_count")
          .orderBy(F.desc("row_count")))

In [ ]:
print("=== Relationship candidates (containment ≥ MIN_CONTAINMENT) ===")
display(rp.where("is_candidate").select("child_table", "child_column", "parent_table", "parent_column", "match_reason", "cardinality",
                                        "containment_rows", "orphan_rows", "orphan_distinct", "child_null_pct", "parent_key_is_unique")
          .orderBy("child_table", "child_column"))
print("   containment_rows < 1.0 means some child rows have no parent (they will map to the Unknown member).")
print("   parent_key_is_unique = false means the parent has duplicate key rows — the build notebook de-duplicates, but check why.")
print("=== Near-misses (name matched but too many orphans) — usually data quality issues, sometimes a wrong parent ===")
display(rp.where("NOT is_candidate AND match_reason <> 'value_scan'").select("child_table", "child_column", "parent_table", "parent_column",
                                                                              "containment_rows", "orphan_rows", "orphan_distinct", "child_distinct")
          .orderBy(F.desc("containment_rows")))

In [ ]:
print("=== Data quality watch-list ===")
issues = (cp.select("table_name", "column_name", "data_type", "semantic_role", "null_pct", "distinct_count", "empty_string_count", "negative_count", "is_constant")
            .where("null_pct >= 50 OR is_constant OR empty_string_count > 0 OR negative_count > 0")
            .withColumn("issue", F.concat_ws("; ",
                F.when(F.col("null_pct") >= 50, F.concat(F.lit("nulls "), F.col("null_pct"), F.lit("%"))),
                F.when(F.col("is_constant"), F.lit("constant value")),
                F.when(F.col("empty_string_count") > 0, F.concat(F.lit("empty strings "), F.col("empty_string_count"))),
                F.when(F.col("negative_count") > 0, F.concat(F.lit("negatives "), F.col("negative_count"))))))
display(issues.orderBy(F.desc("null_pct")))

print("=== Duplicate natural keys: identifier columns that look like a key but are not unique ===")
display(cp.where("semantic_role = 'identifier' AND uniqueness_ratio > 0.9 AND NOT is_candidate_key")
          .select("table_name", "column_name", "row_count", "non_null_count", "distinct_count", "uniqueness_ratio"))

In [ ]:
print("=== Column roles per table (what the modelling notebook will see) ===")
display(cp.groupBy("schema_name", "table_name").pivot("semantic_role", ["key", "identifier", "date", "measure", "flag", "category", "text", "attribute"])
          .count().na.fill(0).orderBy("schema_name", "table_name"))
print("=== Full column profile ===")
display(cp.orderBy("schema_name", "table_name", "ordinal"))

### Next step
Run **`02_logical_model_design`** with `PROFILING_SCHEMA` pointing at the schema above (it defaults to the latest `run_id`).